In [1]:
!pip install roboflow ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [7]:
# Import required packages
import os
import torch
import yaml
from roboflow import Roboflow
from ultralytics import YOLO

In [3]:
# Verify T4 GPU is active
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Device: Tesla T4


#### Load Training Dataset

In [5]:
# Initialize Roboflow API (Replace "YOUR_API_KEY" with your actual Roboflow API key)
rf = Roboflow(api_key="XEMSLJmkPWZDu9NbJ01V")

# Access the project and download YOLOv8 formatted dataset
project = rf.workspace("joseph-nelson").project("plantdoc")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to PlantDoc-1 in yolov8:: 100%|██████████| 5143/5143 [00:00<00:00, 7012.04it/s]


In [6]:
# Define dataset location
dataset_dir = "/content/PlantDoc-1"
yaml_path = os.path.join(dataset_dir, "data.yaml")

print("Directories in dataset:", os.listdir(dataset_dir))

Directories in dataset: ['train', 'data.yaml', 'README.dataset.txt', 'test', 'README.roboflow.txt']


In [8]:
# Repair missing folers in directory
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = dataset_dir
data_config['train'] = 'train/images'

if os.path.exists(os.path.join(dataset_dir, 'valid')):
    data_config['val'] = 'valid/images'
else:
    data_config['val'] = 'test/images'

if os.path.exists(os.path.join(dataset_dir, 'test')):
    data_config['test'] = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print("\n--- Updated data.yaml Config ---")
print(data_config)


--- Updated data.yaml Config ---
{'names': ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf spot', 'Bell_pepper leaf', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Potato leaf', 'Raspberry leaf', 'Soyabean leaf', 'Soybean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato leaf', 'Tomato mold leaf', 'Tomato two spotted spider mites leaf', 'grape leaf black rot', 'grape leaf'], 'nc': 30, 'roboflow': {'license': 'CC BY 4.0', 'project': 'plantdoc', 'url': 'https://universe.roboflow.com/joseph-nelson/plantdoc/dataset/1', 'version': 1, 'workspace': 'joseph-nelson'}, 'test': 'test/images', 'train': 'train/images', 'val': 'test/images', 'path': '/content/PlantDoc-1'}


#### Train custom YOLOv8 model

In [11]:
# Load pre-trained YOLOv8 Nano model
base_model = YOLO("yolov8n.pt")

In [12]:
# Train the model
plant_disease_model = base_model.train(
    data=yaml_path,
    epochs=50,
    imgsz=416,
    batch=16,
    device=0,
    workers=2,
    project="plant_disease_detection",
    name="yolov8n_custom_model"
)

Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PlantDoc-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_custom_model, nbs=64, nms=None, opset=

#### Download the trained model

In [16]:
from google.colab import files
model_path = "/content/runs/detect/plant_disease_detection/yolov8n_custom_model/weights/best.pt"
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>